# 🏷️ Мониторинг цен конкурентов

**Среда выполнения → Выполнить всё**

Собирает ВСЕ товары с 3 сайтов, сопоставляет с твоими SKU, сохраняет в XLSX.

## Шаг 1: Установка

In [ ]:
# Python-пакеты
!pip install -q httpx lxml openpyxl playwright

# Системные библиотеки для Chromium (нужны в Colab Linux)
!apt-get update -qq 2>/dev/null
!apt-get install -y -qq xvfb 2>/dev/null
!playwright install-deps chromium 2>&1 | tail -3

# Сам Chromium
!playwright install chromium 2>&1 | tail -3

print("Готово")

## Шаг 2: Загрузка проекта

In [ ]:
import os

# Всегда начинаем с корня Colab — убираем любые вложенности
%cd /content

# Удаляем старый клон если есть и клонируем заново
!rm -rf price-monitor 2>/dev/null
!git clone -q https://github.com/tswtim/price-monitor.git
%cd /content/price-monitor

# Проверка
!pwd
!ls monitor/__init__.py && echo "OK"

## Шаг 3: Мои SKU

Если файла `my_skus.xlsx` нет — создастся шаблон. Загрузи свой файл сюда же и назови `my_skus.xlsx`.

In [ ]:
import os
if not os.path.exists("my_skus.xlsx"):
    !python -m monitor.cli --init-skus
    print("Создан шаблон my_skus.xlsx. Заполни его своими товарами и запусти заново.")
else:
    from openpyxl import load_workbook
    wb = load_workbook("my_skus.xlsx")
    count = wb.active.max_row - 1
    print(f"Найдено {count} SKU")

## Шаг 4: Запуск

Сбор всех товаров → сопоставление с SKU → result.xlsx

Для delikateska.ru нужен виртуальный дисплей — запускаем Xvfb.

In [ ]:
# Запускаем виртуальный дисплей (нужен для delikateska.ru)
# Xvfb = X Virtual Framebuffer — "фейковый" экран для браузера
import os, subprocess
try:
    subprocess.run(["Xvfb", ":99", "-screen", "0", "1280x1024x24"], 
                   check=False, capture_output=True, 
                   start_new_session=True)
    os.environ["DISPLAY"] = ":99"
    print("Xvfb запущен (DISPLAY=:99)")
except:
    print("Xvfb не нужен (Windows?)")

# Запуск мониторинга
!python -m monitor.cli --run

## Шаг 5: Результат

**Скачай `result.xlsx`** — слева в панели файлов, правый клик → Скачать.

### Листы в result.xlsx
- **Совпадения** — все продукты × SKU. Ставь `да`/`нет` в колонке `is_comparable`
- **Сводка по SKU** — мин/средняя/макс цена по каждому твоему товару
- **Мои SKU** — твои товары с ключевыми словами

### Как добавлять свои ссылки
Добавь строку в лист «Совпадения»: вставь URL, `matched_sku`, `is_comparable="да"`.
При следующем запуске скрипт сам сходит по ссылке и обновит цену.